# Example 10.2.1 / 2: Find a Shortest Path

Companion notebook for **10.2.1 Maximum Flow and Shortest Path** in the AMPL lab Documentation, section "Find a Shortest Path". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Find a Shortest Path

Use the balance formulation from Network Models (AMPL lab documentation, section 10.1) with `balance[source] = 1`, `balance[sink] = -1`, zero elsewhere, and arc length as the cost. Positive-length cycles cannot improve the objective; arcs carrying one unit identify a shortest path.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES;                                   # network nodes
param source symbolic in NODES;              # start of the path
param sink symbolic in NODES, != source;     # end of the path
set ARCS within NODES cross NODES;           # directed roads
param length {ARCS} >= 0;                    # length of each road

var Use {ARCS} >= 0;                         # 1 if the arc is on the path

minimize TotalLength: sum {(i, j) in ARCS} length[i, j] * Use[i, j];

subject to Balance {k in NODES}:             # one unit leaves source, one reaches sink
    sum {(k, j) in ARCS} Use[k, j] - sum {(i, k) in ARCS} Use[i, k]
  = if k = source then 1 else if k = sink then -1 else 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NODES"] = ["s", "a", "b", "c", "t"]
ampl.param["source"] = "s"
ampl.param["sink"] = "t"
length = {("s", "a"): 2, ("s", "b"): 5, ("a", "b"): 1, ("a", "c"): 6,
          ("b", "c"): 2, ("b", "t"): 7, ("c", "t"): 1}
ampl.set["ARCS"] = list(length)
ampl.param["length"] = length

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["TotalLength"].value())
use = ampl.var["Use"].to_dict()
print([arc for arc, value in use.items() if value > 0.5])

solved 6.0
[('a', 'b'), ('b', 'c'), ('c', 't'), ('s', 'a')]


### Pure AMPL: data.dat and model.run

The shortest route is `s -> a -> b -> c -> t` with length 2 + 1 + 2 + 1 = 6, which matches the
objective; the four arcs with value 1 form the route.

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set NODES := s a b c t;
param source := s;
param sink := t;
param: ARCS: length :=
  s a 2   s b 5   a b 1   a c 6   b c 2   b t 7   c t 1 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display TotalLength;
display {(i, j) in ARCS: Use[i, j] > 0.5} Use[i, j];

TotalLength = 6

Use[i,j] :=
a b   1
b c   1
c t   1
s a   1
;

